### Random Forest Classifier to Predict a Successful Vaginal Birth After C-Section

Now that we've created the most basic prediction model, we'll test a random forest classifier to see if it achieves better accuracy.

In [ ]:
from pickle import load

x_train_unfiltered = load(
    open("../../../data/processed/vbac/X_train_prepared.pkl", "rb")
)
x_test_unfiltered = load(open("../../../data/processed/vbac/X_test_prepared.pkl", "rb"))
feature_importances_df = load(
    open("../../../data/processed/vbac/feature_importances_df.pkl", "rb")
)

In [ ]:
y_test = load(open("../../../data/processed/vbac/y_test.pkl", "rb"))
y_train = load(open("../../../data/processed/vbac/y_train.pkl", "rb"))

We'll use our `feature_importances` dataframe to train a few versions of the model with different configurations of the data.

First, we'll try training using any feature with an importance > 0.01.

In [ ]:
x_train_001 = x_train_unfiltered[
    :,
    feature_importances_df.loc[
        feature_importances_df["importance"] > 0.01
    ].index.tolist(),
]
x_test_001 = x_test_unfiltered[
    :,
    feature_importances_df.loc[
        feature_importances_df["importance"] > 0.01
    ].index.tolist(),
]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc_model_001 = RandomForestClassifier(random_state=14, class_weight="balanced")

rfc_model_001.fit(x_train_001, y_train)

In [ ]:
predictions_001 = rfc_model_001.predict(x_test_001)

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, predictions_001)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=rfc_model_001.classes_
)
disp.plot()
plt.show()

In [ ]:
from sklearn.metrics import f1_score

f1_001 = f1_score(y_test, predictions_001)

f1_001

Next we'll train a model with features of importance > 0.15.

In [ ]:
x_train_0015 = x_train_unfiltered[
    :,
    feature_importances_df.loc[
        feature_importances_df["importance"] > 0.015
    ].index.tolist(),
]
x_test_0015 = x_test_unfiltered[
    :,
    feature_importances_df.loc[
        feature_importances_df["importance"] > 0.015
    ].index.tolist(),
]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc_model_0015 = RandomForestClassifier(random_state=14, class_weight="balanced")

rfc_model_0015.fit(x_train_0015, y_train)

In [ ]:
predictions_0015 = rfc_model_0015.predict(x_test_0015)

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, predictions_0015)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=rfc_model_0015.classes_
)
disp.plot()
plt.show()

In [ ]:
from sklearn.metrics import f1_score

f1_0015 = f1_score(y_test, predictions_0015)

f1_0015

Next we'll train a model with features of importance > 0.2.

In [ ]:
x_train_002 = x_train_unfiltered[
    :,
    feature_importances_df.loc[
        feature_importances_df["importance"] > 0.02
    ].index.tolist(),
]
x_test_002 = x_test_unfiltered[
    :,
    feature_importances_df.loc[
        feature_importances_df["importance"] > 0.02
    ].index.tolist(),
]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc_model_002 = RandomForestClassifier(random_state=14, class_weight="balanced")

rfc_model_002.fit(x_train_002, y_train)

In [ ]:
predictions_002 = rfc_model_002.predict(x_test_002)

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, predictions_002)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=rfc_model_002.classes_
)
disp.plot()
plt.show()

In [ ]:
from sklearn.metrics import f1_score

f1_002 = f1_score(y_test, predictions_002)

f1_002

Next we'll train a model with a subset of the features based on domain knowledge.

In [ ]:
custom_filtered_features_df = feature_importances_df.loc[
    feature_importances_df["importance"] > 0.01
].copy()

custom_filtered_features_df

Some of these features are duplicates of each other, or otherwise highly correlated. First, we'll clean those up.

In [ ]:
custom_filtered_features_df = custom_filtered_features_df.drop([1597, 1550, 1514])

Although paternal age plays a factor in fertility and pregnancy, we don't believe it to play a factor in VBAC. We'll rely on the mother's age instead of both ages.

In [ ]:
custom_filtered_features_df = custom_filtered_features_df.drop([1587])

We can now train our model.

In [ ]:
x_train_custom = x_train_unfiltered.iloc[
    :,
    custom_filtered_features_df.index.tolist(),
].copy()
x_test_custom = x_test_unfiltered.iloc[
    :,
    custom_filtered_features_df.index.tolist(),
].copy()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc_model_custom = RandomForestClassifier(random_state=14, class_weight="balanced")

rfc_model_custom.fit(x_train_custom, y_train)

In [ ]:
predictions_custom = rfc_model_custom.predict(x_test_custom)

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, predictions_custom)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=rfc_model_custom.classes_
)
disp.plot()
plt.show()

In [ ]:
from sklearn.metrics import f1_score

f1_custom = f1_score(y_test, predictions_custom)

f1_custom

### Findings

In [ ]:
print("Importance 0.01 F1 score: ", f1_001)
print("Importance 0.015 F1 score: ", f1_0015)
print("Importance 0.02 F1 score: ", f1_002)
print("Custom filter F1 score: ", f1_custom)

The 0.01 and custom models performed at approximately the same level. We will move forward with the custom model, as its smaller feature set will be easier to implement in an interactive format.

We'll create a new preprocessing pipeline targeting only the features we need.

In [ ]:
import pandas as pd
from pickle import load

X_train = load(open("../../../data/interim/vbac/X_train.pkl", "rb"))
X_test = load(open("../../../data/interim/vbac/X_test.pkl", "rb"))

In [ ]:
cat_features = [
    "augmentation_of_labor",
    "induction_of_labor",
    "attendant_at_birth",
]

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline

cat_pipeline = make_pipeline(
    SimpleImputer(missing_values=pd.NA, strategy="most_frequent"),
    OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
)

num_features = [
    "prior_births_now_living",
    "time_of_birth",
    "mothers_single_year_age",
    "interval_since_last_live_birth",
    "number_of_prenatal_visits",
    "BMI",
    "weight_gain",
    "number_of_previous_cesarean",
    "combined_gestation_detail_in_weeks",
    "birth_weight_in_grams",
]

from sklearn.preprocessing import StandardScaler

num_pipeline = make_pipeline(
    SimpleImputer(missing_values=pd.NA, strategy="mean"), StandardScaler()
)

from sklearn.compose import ColumnTransformer

preprocessing_final = ColumnTransformer(
    [
        ("Categorical Pipeline", cat_pipeline, cat_features),
        ("Numerical Pipeline", num_pipeline, num_features),
    ]
)

preprocessing_final.set_output(transform="pandas")

preprocessing_final.fit(X_train)

Now we'll train our model with the new pipeline

In [ ]:
X_train_final = preprocessing_final.transform(X_train)
X_test_final = preprocessing_final.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc_model_final = RandomForestClassifier(random_state=14, class_weight="balanced")

rfc_model_final.fit(X_train_final, y_train)

In [ ]:
predictions_final = rfc_model_final.predict(X_test_final)

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, predictions_final)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=rfc_model_final.classes_
)
disp.plot()
plt.show()

In [ ]:
from sklearn.metrics import f1_score

f1_final = f1_score(y_test, predictions_final)

f1_final

In [ ]:
from pickle import dump

dump(preprocessing_final, open("../../../models/vbac/preprocessing_pipeline.pkl", "wb"))
dump(rfc_model_final, open("../../../models/vbac/rfc_model.pkl", "wb"))

### Conclusion

Our RFC model is able to predict VBAC outcomes with moderate accuracy. A more complex model may have greater success at accurate predictions.